# Malignant and Macrophage Interactions Across Treatment Groups (TN, OT, TKIR)

**Goal.** Compare selected recurrent ligand–receptor interactions between malignant states and
macrophage states across treatment-naive (TN), on-treatment (OT) and TKI-resistant (TKIR) tumours.

**Input.** Per group, `consensus_compartments_0.5_strict.h5ad`: directed cell pairs ×
interactions; `means` = mean across patients, `fraction_significant` = fraction of evaluable patients
with P < 0.05 (slots < 0.5 set to 0).

**Figure.** 2 × 3 dot plot (columns = groups), interactions grouped by biological theme.
Dot size = fraction of patients significant. Row 1: consensus mean, one colour scale shared by all
groups. Row 2: mean scaled 0–1 per interaction within each group; * = scaled ≥ 0.9 (the dominant
cell pair(s) for that interaction in that group).

**Output.** `malignant_macrophage_TN_OT_TKIR_dotplot.pdf` + source table.

In [ ]:
# !pip install --quiet anndata seaborn
# %cd <project>/cpdb/cellphoneDB_v5_plot

In [ ]:
import os
import re

import anndata as ad
import matplotlib.patheffects as pe
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

## Config

In [ ]:
base_dir = "."                          # contains <group>/cpdb_adata_per_patient/
groups = ["TN", "OT", "TKIR"]
consensus_file = "cpdb_adata_per_patient/consensus_compartments_0.5_strict.h5ad"
out_dir = "./plots_malignant_macrophage"

malignant_states = ["3_Devp_stem-like", "4_Inflammatory EMT", "5_Neurod_EMT-like", "6_Proliferative"]
macrophage_states = ["Macrophage INHBAhigh", "Macrophage STAB1high", "Macrophage CD36high"]

os.makedirs(out_dir, exist_ok=True)

## 1. Collect consensus slots per group

All interactions matching `themes` are kept (no top-N cut), so a missing dot means the slot did not
pass the consensus filter in that group, not that it fell outside a ranking.

In [ ]:
# interactions shown: any interaction whose partners include one of these names (see evaluation plots)
# grouped by theme in this order; names not in a theme go to "Other"
themes = {
    "Immune evasion": ["CD47", "LGALS9", "CD55", "CD99"],
    "Growth / survival": ["TGFA", "EGFR", "VEGFA"],
    "Stemness / plasticity": ["WNT2B", "TGFB1", "JAG1", "NOTCH2"],
    "Phagocytosis / efferocytosis": ["GAS6", "MERTK"],
    "Adhesion": ["ICAM1"],
    "Other": ["LGALS3", "APP", "CXCL2", "CLSTN1", "CLSTN2", "CLSTN3", "Glutamate", "Cholesterol",
              "BMP5", "ADGRE2", "TNFSF10"],
}

name_to_theme = {n: t for t, names in themes.items() for n in names}
pair_order = ([f"{m}|{k}" for m in malignant_states for k in macrophage_states]
              + [f"{k}|{m}" for k in macrophage_states for m in malignant_states])


def match_theme(pair_name):
    tokens = set(re.split(r"[_\s]", pair_name))
    hits = [name_to_theme[t] for t in tokens if t in name_to_theme]
    return min(hits, key=list(themes).index) if hits else None


def collect(group):
    a = ad.read_h5ad(os.path.join(base_dir, group, consensus_file))
    s, r = a.obs["sender"], a.obs["receiver"]
    keep = ((s.isin(malignant_states) & r.isin(macrophage_states))
            | (s.isin(macrophage_states) & r.isin(malignant_states))).to_numpy()
    a = a[keep]
    df = (pd.DataFrame(a.layers["means"], index=a.obs_names, columns=a.var_names).stack().rename("mean_score")
          .to_frame().join(pd.DataFrame(a.layers["fraction_significant"], index=a.obs_names,
                                        columns=a.var_names).stack().rename("frac_sig"))
          .rename_axis(["cell_pair", "interaction"]).reset_index())
    df["interaction_label"] = df["interaction"].map(a.var["interacting_pair"]).astype(str)
    df["theme"] = df["interaction_label"].map(match_theme)
    df = df[(df["frac_sig"] > 0) & df["theme"].notna()].copy()
    df["group"] = group
    return df


plot_df = pd.concat([collect(g) for g in groups], ignore_index=True)

# star column: min-max of mean per interaction within each group (a single dot = 1)
g = plot_df.groupby(["group", "interaction_label"])["mean_score"]
span = g.transform("max") - g.transform("min")
plot_df["scaled"] = np.where(span > 0, (plot_df["mean_score"] - g.transform("min")) / span.where(span > 0, 1), 1.0)
plot_df["star"] = plot_df["scaled"] >= star_threshold
plot_df.to_csv(f"{out_dir}/malignant_macrophage_TN_OT_TKIR_source.csv", index=False)

pd.crosstab(plot_df["theme"], plot_df["group"]).reindex(index=list(themes), columns=groups).fillna(0).astype(int)

## 2. Dot plot merged

Interactions ordered by theme, then name; numeric positions keep x/y labels aligned with the dots
in every panel.

In [ ]:
star_threshold = 0.9                    # scaled score marking the dominant cell pair(s) per interaction
dot_scale = 200                         # marker area at fraction_significant = 1
size_legend = [0.5, 0.75, 1.0]          # consensus input is filtered at >= 0.5
cmap_raw, cmap_scaled = "RdPu", "viridis"

rows = (plot_df[["theme", "interaction_label"]].drop_duplicates()
        .assign(order=lambda d: d["theme"].map(list(themes).index))
        .sort_values(["order", "interaction_label"]))
y_labels = rows["interaction_label"].tolist()
y_pos = {l: i for i, l in enumerate(y_labels)}
x_pos = {p: i for i, p in enumerate(pair_order)}
vmax_raw = plot_df["mean_score"].max()
n_mal = len(malignant_states) * len(macrophage_states)

fig, axes = plt.subplots(2, len(groups), figsize=(8 * len(groups), 0.42 * len(y_labels) * 2 + 6),
                         sharex=True, sharey=True)
for col, grp in enumerate(groups):
    df = plot_df[plot_df["group"] == grp]
    x, y = df["cell_pair"].map(x_pos), df["interaction_label"].map(y_pos)
    for row, (colour, cmap, vmax, label) in enumerate([
            ("mean_score", cmap_raw, vmax_raw, "Consensus mean score"),
            ("scaled", cmap_scaled, 1, "Mean scaled per interaction")]):
        ax = axes[row, col]
        h = ax.scatter(x, y, s=df["frac_sig"] * dot_scale, c=df[colour], cmap=cmap, vmin=0, vmax=vmax,
                       edgecolors="black", linewidths=0.1)
        if row == 1:
            for xi, yi in zip(x[df["star"]], y[df["star"]]):
                ax.text(xi, yi, "*", ha="center", va="center", fontsize=14, zorder=15,
                        path_effects=[pe.withStroke(linewidth=1, foreground="black")])
        ax.axvline(n_mal - 0.5, color="grey", ls="--", lw=0.8)
        ax.set_title(f"{grp}: {'raw' if row == 0 else 'scaled per interaction'}")
        cb = plt.colorbar(h, cax=inset_axes(ax, width="3%", height="20%", loc="upper right"))
        cb.set_label(label, fontsize=8)

for ax in axes.flat:
    ax.set_xticks(range(len(pair_order)), pair_order, rotation=90, fontsize=12)
    ax.set_yticks(range(len(y_labels)), y_labels, fontstyle="italic")
    ax.set_xlim(-0.5, len(pair_order) - 0.5)
    ax.set_ylim(len(y_labels) - 0.5, -0.5)
for ax in axes[:, 0]:
    ax.set_ylabel("Ligand–receptor interaction")

handles = [plt.scatter([], [], s=v * dot_scale, color="grey", edgecolors="black") for v in size_legend]
fig.legend(handles, [f"{v:.2f}" for v in size_legend], title="Fraction of patients\nsignificant",
           loc="upper right", bbox_to_anchor=(0.995, 0.995), frameon=False)
plt.tight_layout()
plt.savefig(f"{out_dir}/malignant_macrophage_TN_OT_TKIR_dotplot.pdf", bbox_inches="tight") #Figure 6
plt.show()